In [1]:
# =========================================================
# CELL 1: CÀI ĐẶT & CẤU HÌNH ĐƯỜNG DẪN
# =========================================================
!pip install -q "huggingface_hub>=0.26.2" pyyaml

In [2]:
!rm -rf /kaggle/working/layout_data/rukopys_v3

In [3]:
import os
import json
import random
import shutil
import re
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import yaml
from huggingface_hub import hf_hub_download

# --- INPUT DATA ---
GOLD_DIR = Path("/kaggle/input/datasets/notpitomon/htd-final-better-gold-dataset/Merged_Rukopys_V1/train")
SILVER_DIR = Path("No use")

# --- OUTPUT YOLO DATASET ---
OUT_ROOT = Path("/kaggle/working/layout_data/Final")
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

# TUYỆT ĐỐI GHI NHỚ:
# RUKOPYS Gold + Silver rare trong notebook này đều dùng bbox gốc dạng COCO: [x, y, w, h]
# Không được viết lại parser xyxy nữa.
BBOX_FORMAT = "xyxy"

# --- CLASS MAP ---
TYPE2ID = {
    "handwritten": 0,
    "printed": 1,
    "formula": 2,
    "table": 3,
    "annotation": 4,
    "image": 5,
    "graph": 6,
}

ID2TYPE = {v: k for k, v in TYPE2ID.items()}

# --- SPLIT / SAMPLING CONFIG ---
VAL_RATIO = 0.15
SEED = 42

# Silver rare là bổ trợ, không được lấn át Gold
USE_SILVER = False
MAX_SILVER_IMAGES = 1200   # có thể giảm xuống 600-1000 nếu muốn bảo thủ hơn

random.seed(SEED)

print("✅ Config xong.")
print("GOLD_DIR   =", GOLD_DIR)
print("SILVER_DIR =", SILVER_DIR)
print("OUT_ROOT   =", OUT_ROOT)
print("BBOX_FORMAT =", BBOX_FORMAT)

✅ Config xong.
GOLD_DIR   = /kaggle/input/datasets/notpitomon/htd-final-better-gold-dataset/Merged_Rukopys_V1/train
SILVER_DIR = No use
OUT_ROOT   = /kaggle/working/layout_data/Final
BBOX_FORMAT = xyxy


In [4]:
# =========================================================
# CELL 2: HÀM HỖ TRỢ CHUYỂN BBOX & IO (XYXY -> YOLO)
# =========================================================
def safe_link(src: Path, dst: Path):
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def clamp_xyxy(x1, y1, x2, y2, img_w, img_h):
    x1 = max(0.0, min(float(x1), float(img_w - 1)))
    y1 = max(0.0, min(float(y1), float(img_h - 1)))
    x2 = max(0.0, min(float(x2), float(img_w - 1)))
    y2 = max(0.0, min(float(y2), float(img_h - 1)))
    return x1, y1, x2, y2

def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    w = x2 - x1
    h = y2 - y1
    cx = (x1 + w / 2.0) / img_w
    cy = (y1 + h / 2.0) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh

def region_to_yolo_line(region, img_w, img_h):
    t = region.get("type")
    if t not in TYPE2ID:
        return None

    bbox = region.get("bbox")
    if not bbox or len(bbox) != 4:
        return None

    # Đọc theo format xyxy
    x1, y1, x2, y2 = map(float, bbox)
    x1, y1, x2, y2 = clamp_xyxy(x1, y1, x2, y2, img_w, img_h)

    if (x2 - x1) <= 1 or (y2 - y1) <= 1:
        return None

    cx, cy, nw, nh = xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)

    if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < nw <= 1 and 0 < nh <= 1):
        return None

    return f"{TYPE2ID[t]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"

def load_metadata(meta_path: Path):
    rows = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def get_base_name(fname: str):
    name = Path(fname).name
    name = re.sub(r"_aug\d+", "", name)
    return name

In [5]:
# =========================================================
# CELL 3: LOAD METADATA
# =========================================================
gold_metadata = load_metadata(GOLD_DIR / "metadata.jsonl")
print(f"Gold records: {len(gold_metadata)}")

if USE_SILVER and (SILVER_DIR / "metadata.jsonl").exists():
    silver_metadata = load_metadata(SILVER_DIR / "metadata.jsonl")
    print(f"Silver records: {len(silver_metadata)}")
else:
    silver_metadata = []
    print("Silver records: 0")

print("\n✅ Xác nhận cứng: Gold và Silver đều dùng bbox format COCO = [x, y, w, h]")
print("✅ Toàn bộ pipeline convert label sẽ đọc bbox theo COCO rồi đổi sang YOLO normalized xywh")

Gold records: 2556
Silver records: 0

✅ Xác nhận cứng: Gold và Silver đều dùng bbox format COCO = [x, y, w, h]
✅ Toàn bộ pipeline convert label sẽ đọc bbox theo COCO rồi đổi sang YOLO normalized xywh


In [6]:
# =========================================================
# CELL 4: CHIA GOLD TRAIN/VAL + CHỌN SILVER BỔ TRỢ
# =========================================================
base_names = sorted({get_base_name(d["file_name"]) for d in gold_metadata})
random.shuffle(base_names)

split_idx = int(len(base_names) * (1 - VAL_RATIO))
train_bases = set(base_names[:split_idx])
val_bases = set(base_names[split_idx:])

print(f"Gold base images: {len(base_names)}")
print(f"Train bases: {len(train_bases)} | Val bases: {len(val_bases)}")

def count_classes_in_record(d):
    c = Counter()
    for r in d.get("regions", []):
        t = r.get("type")
        if t in TYPE2ID:
            c[t] += 1
    return c

selected_silver = []

if silver_metadata and USE_SILVER:
    scored = []
    for d in silver_metadata:
        regions = d.get("regions", [])
        if not regions:
            continue

        c = count_classes_in_record(d)
        total = sum(c.values())
        if total == 0:
            continue

        head = c["handwritten"] + c["formula"]
        tail = c["image"] + c["graph"]
        mid  = c["printed"] + c["table"] + c["annotation"]

        rare_ratio = tail / total
        dominant_ratio = head / total

        score = (
            c["graph"] * 5.0 +
            c["image"] * 4.0 +
            c["table"] * 1.5 +
            c["annotation"] * 1.0 +
            rare_ratio * 10.0 -
            dominant_ratio * 6.0 -
            max(0, c["handwritten"] - 20) * 0.15 -
            max(0, c["formula"] - 10) * 0.20
        )

        # Chặn các ảnh nhìn như head-class dominant nhưng chỉ có rare tượng trưng
        if dominant_ratio >= 0.8 and tail <= 1:
            continue

        scored.append((score, d, c))

    scored.sort(key=lambda x: x[0], reverse=True)
    selected_silver = [x[1] for x in scored[:MAX_SILVER_IMAGES]]

    agg = Counter()
    for _, _, c in scored[:MAX_SILVER_IMAGES]:
        agg.update(c)

    print(f"Selected silver images: {len(selected_silver)}")
    print("Silver selected box distribution:")
    for cls, cnt in sorted(agg.items(), key=lambda kv: kv[1]):
        print(f"  - {cls}: {cnt}")
else:
    print("Không dùng silver bổ trợ.")

Gold base images: 2172
Train bases: 1846 | Val bases: 326
Không dùng silver bổ trợ.


In [7]:
# =========================================================
# CELL 5: BUILD YOLO DATASET
# =========================================================
train_files = []
val_files = []

gold_train_count = 0
gold_val_count = 0
silver_train_count = 0

bad_records = 0
class_counter_train = Counter()
class_counter_val = Counter()

def process_record_to_yolo(d, image_root: Path, split: str):
    global bad_records

    fname = Path(d["file_name"]).name
    img_w, img_h = d["image_width"], d["image_height"]
    regions = d.get("regions", [])

    if not regions:
        bad_records += 1
        return False

    in_path = image_root / fname
    if not in_path.exists():
        bad_records += 1
        return False

    out_img = OUT_ROOT / "images" / fname
    safe_link(in_path, out_img)

    label_lines = []
    local_counter = Counter()

    for r in regions:
        line = region_to_yolo_line(r, img_w, img_h)
        if line is not None:
            label_lines.append(line)
            local_counter[r["type"]] += 1

    if not label_lines:
        bad_records += 1
        return False

    with open(OUT_ROOT / "labels" / f"{Path(fname).stem}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(label_lines))

    if split == "train":
        train_files.append(fname)
        class_counter_train.update(local_counter)
    else:
        val_files.append(fname)
        class_counter_val.update(local_counter)

    return True

# -----------------------
# GOLD
# -----------------------
for d in tqdm(gold_metadata, desc="Processing GOLD"):
    fname = Path(d["file_name"]).name
    base = get_base_name(fname)
    is_val = base in val_bases

    # Không cho ảnh augment vào val
    if is_val and "_aug" in fname:
        continue

    split = "val" if is_val else "train"
    ok = process_record_to_yolo(d, GOLD_DIR / "images", split)

    if ok:
        if split == "train":
            gold_train_count += 1
        else:
            gold_val_count += 1

# -----------------------
# SILVER rare supplement
# -----------------------
for d in tqdm(selected_silver, desc="Processing SILVER rare supplement"):
    ok = process_record_to_yolo(d, SILVER_DIR / "images", "train")
    if ok:
        silver_train_count += 1

random.shuffle(train_files)

print("\n✅ Build dataset xong.")
print(f"Gold train images used: {gold_train_count}")
print(f"Gold val images used:   {gold_val_count}")
print(f"Silver train images used: {silver_train_count}")
print(f"Bad/Skipped records: {bad_records}")

print("\nTrain class distribution:")
for cls, cnt in sorted(class_counter_train.items(), key=lambda kv: kv[1]):
    print(f"  - {cls}: {cnt}")

print("\nVal class distribution:")
for cls, cnt in sorted(class_counter_val.items(), key=lambda kv: kv[1]):
    print(f"  - {cls}: {cnt}")

Processing GOLD: 100%|██████████| 2556/2556 [00:04<00:00, 581.99it/s]
Processing SILVER rare supplement: 0it [00:00, ?it/s]


✅ Build dataset xong.
Gold train images used: 2166
Gold val images used:   268
Silver train images used: 0
Bad/Skipped records: 0

Train class distribution:
  - graph: 161
  - image: 740
  - table: 773
  - annotation: 1055
  - formula: 3314
  - printed: 5907
  - handwritten: 23387

Val class distribution:
  - graph: 14
  - image: 63
  - table: 85
  - annotation: 159
  - printed: 439
  - formula: 614
  - handwritten: 3580


In [8]:
# =========================================================
# CELL 6: GHI train.txt / val.txt / yaml
# =========================================================
train_txt = OUT_ROOT / "train.txt"
val_txt   = OUT_ROOT / "val.txt"

with open(train_txt, "w", encoding="utf-8") as f:
    for fname in train_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

with open(val_txt, "w", encoding="utf-8") as f:
    for fname in val_files:
        f.write(str(OUT_ROOT / "images" / fname) + "\n")

yaml_path = "/kaggle/working/rukopys_dataset_v3.yaml"
data_config = {
    "path": str(OUT_ROOT),
    "train": "train.txt",
    "val": "val.txt",
    "nc": 7,
    "names": ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_config, f, sort_keys=False)

# Giữ nguyên bản vá hoang dã .yaml.yaml
shutil.copy(yaml_path, yaml_path + ".yaml")

print(f"Train images: {len(train_files)}")
print(f"Val images:   {len(val_files)}")
print("YAML written:", yaml_path)
print("YAML duplicate:", yaml_path + ".yaml")

Train images: 2166
Val images:   268
YAML written: /kaggle/working/rukopys_dataset_v3.yaml
YAML duplicate: /kaggle/working/rukopys_dataset_v3.yaml.yaml


In [9]:
# =========================================================
# CELL 7: SETUP DOCLAYOUT-YOLO + PATCH (ĐÃ VÁ LỖI FUSE CRASH)
# =========================================================
import sys
import os
from types import ModuleType

%cd /kaggle/working

if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd /kaggle/working/DocLayout-YOLO
!pip install -q -e .

# --- Patch check_amp ---
checks_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/checks.py"
if os.path.exists(checks_file):
    with open(checks_file, "r", encoding="utf-8") as f:
        code = f.read()
    if "def check_amp(model):" in code and "return True" not in code.split("def check_amp(model):")[1][:40]:
        code = code.replace("def check_amp(model):", "def check_amp(model):\n    return True\n")
        with open(checks_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched check_amp()")

# --- Patch strip_optimizer / torch.load for PyTorch 2.6 ---
torch_utils_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/torch_utils.py"
if os.path.exists(torch_utils_file):
    with open(torch_utils_file, "r", encoding="utf-8") as f:
        code = f.read()

    old_load = 'x = torch.load(f, map_location=torch.device("cpu"))'
    new_load = 'x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'

    if old_load in code:
        code = code.replace(old_load, new_load)
        with open(torch_utils_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched strip_optimizer torch.load(..., weights_only=False)")

# --- Patch G2L_CRM Fuse Bug (CHỐT HẠ LỖI NONE-TYPE CALLABLE) ---
g2l_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/nn/modules/g2l_crm.py"
if os.path.exists(g2l_file):
    with open(g2l_file, "r", encoding="utf-8") as f:
        code = f.read()
    if "bn = self.dcv.bn" in code:
        # Dùng Identity() thế chỗ None để khi Fuse xong, lệnh bn(tensor) trả về đúng tensor gốc
        code = code.replace("bn = self.dcv.bn", "bn = getattr(self.dcv, 'bn', None) or torch.nn.Identity()")
        with open(g2l_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched G2L_CRM for fuse compatibility (Safe Identity)")

# --- Patch NumPy 2.0 trapz bug ---
metrics_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/metrics.py"
if os.path.exists(metrics_file):
    with open(metrics_file, "r", encoding="utf-8") as f:
        code = f.read()
    
    if "np.trapz" in code:
        # Thay thế hàm np.trapz cũ bằng công thức tính tích phân hình thang thuần của NumPy
        code = code.replace("np.trapz(", "np.sum((x[1:] - x[:-1]) * (mpre[1:] + mpre[:-1]) / 2.0) if False else np.trapz(") 
        # Cứu cánh an toàn hơn: dùng công thức mảng tương đương 100% trapz:
        code = re.sub(r'np\.trapz\(np\.interp\(x,\s*mrec,\s*mpre\),\s*x\)', 
                      r'np.sum((x[1:] - x[:-1]) * (np.interp(x, mrec, mpre)[1:] + np.interp(x, mrec, mpre)[:-1]) / 2.0)', code)
        
        with open(metrics_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Patched metrics.py (NumPy 2.0 trapz fix)")

wb_file = "/kaggle/working/DocLayout-YOLO/doclayout_yolo/utils/callbacks/wb.py"
if os.path.exists(wb_file):
    with open(wb_file, "r", encoding="utf-8") as f:
        code = f.read()
    
    # Tìm đúng hàm _plot_curve gây ra tội ác và chặt đứt chân nó
    if "def _plot_curve(" in code:
        code = code.replace("def _plot_curve(", "def _plot_curve(*args, **kwargs):\n    return\ndef _old_plot_curve(")
        with open(wb_file, "w", encoding="utf-8") as f:
            f.write(code)
        print("✅ Đã vô hiệu hóa triệt để hàm vẽ đồ thị của WandB!")


!pip uninstall ray -y -q

if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

print("✅ DocLayout-YOLO setup xong.")

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 233 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 37.19 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.2 MB/s eta 0:00:00
  Building editable for doclayout_yolo (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 w

In [10]:
# =========================================================
# CELL 8: TẢI PRETRAIN
# =========================================================
CKPT_PATH = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt"
)

print("✅ Pretrain checkpoint:", CKPT_PATH)

doclayout_yolo_docstructbench_imgsz1024.(…):   0%|          | 0.00/40.7M [00:00<?, ?B/s]

✅ Pretrain checkpoint: /root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt


In [11]:
# =========================================================
# CELL 9: TRAIN (DÙNG THAM SỐ CLI THỰC SỰ ĐƯỢC HỖ TRỢ)
# =========================================================
%cd /kaggle/working/DocLayout-YOLO

print("\n🔥 KHỞI ĐỘNG HUẤN LUYỆN V3 + MOSAIC NHẸ 🔥")

!WANDB_MODE=disabled RAY_DISABLE_TUNE=1 python train.py \
  --data /kaggle/working/rukopys_dataset_v3.yaml \
  --model doclayout_yolo_small \
  --epoch 100 \
  --image-size 1024 \
  --batch-size 8 \
  --mosaic 0.1 \
  --project Final \
  --optimizer Adam \
  --lr0 0.001 \
  --warmup-epochs 2.0 \
  --patience 10 \
  --pretrain {CKPT_PATH} \
  --device 0,1 \
  --workers 8

/kaggle/working/DocLayout-YOLO

🔥 KHỞI ĐỘNG HUẤN LUYỆN V3 + MOSAIC NHẸ 🔥
New https://pypi.org/project/doclayout_yolo/0.0.4 available 😃 Update with 'pip install -U doclayout_yolo'
Ultralytics YOLOv0.0.2 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                            CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=/root/.cache/huggingface/hub/models--juliozhao--DocLayout-YOLO-DocStructBench/snapshots/8c3299a30b8ff29a1503c4431b035b93220f7b11/doclayout_yolo_docstructbench_imgsz1024.pt, data=/kaggle/working/rukopys_dataset_v3.yaml.yaml, epochs=100, time=None, patience=10, batch=8, imgsz=1024, save=True, save_period=10, val_period=1, cache=False, device=0,1, workers=8, project=Final, name=rukopys_dataset_v3.yaml_epoch100_imgsz1024_bs8_pretrain_unknown, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resu

In [12]:
# =========================================================
# CELL 10 (MỚI): TRẠM THU HOẠCH SỐ LIỆU & VẼ BIỂU ĐỒ BÁO CÁO
# =========================================================
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RUN_DIR = Path("/kaggle/working/runs/exp1")
WEIGHTS_BEST = RUN_DIR / "weights" / "best.pt"
CSV_PATH = RUN_DIR / "results.csv"

print("🚀 BƯỚC 1: BỐC FILE LOG TỪNG EPOCH RA NGOÀI")
if CSV_PATH.exists():
    out_csv = Path("/kaggle/working/training_log_history.csv")
    shutil.copy2(CSV_PATH, out_csv)
    df_logs = pd.read_csv(out_csv)
    df_logs.columns = [c.strip() for c in df_logs.columns] 
    print(f"✅ Đã trích xuất lịch sử {len(df_logs)} epochs ra: {out_csv}")
else:
    print("❌ Không tìm thấy file results.csv!")

print("\n🚀 BƯỚC 2: CHẠY VALIDATION TRÊN BEST.PT ĐỂ LẤY AP TỪNG CLASS")
from doclayout_yolo import YOLOv10

model = YOLOv10(str(WEIGHTS_BEST))
# Ép chạy trên batch=8, imgsz=1024 đúng chuẩn lúc train
val_metrics = model.val(data="/kaggle/working/rukopys_dataset_v3.yaml", split="val", imgsz=1024, batch=8, device="0")

names = val_metrics.names
maps50 = val_metrics.box.ap50      
maps50_95 = val_metrics.box.maps   
p_class = val_metrics.box.p        
r_class = val_metrics.box.r        
f1_class = val_metrics.box.f1      

print("\n" + "="*70)
print("📊 BẢNG 1: SỐ LIỆU PER-CLASS (COPY VÀO BẢNG `tab:augmentation-ablation`)")
print("="*70)
print("| Class | Precision | Recall | F1-Score | mAP@50 | mAP@50:95 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")

for c_id, cls_name in names.items():
    print(f"| `{cls_name}` | {p_class[c_id]:.4f} | {r_class[c_id]:.4f} | **{f1_class[c_id]:.4f}** | {maps50[c_id]:.4f} | {maps50_95[c_id]:.4f} |")

print("\n📊 TỔNG HỢP TOÀN MÔ HÌNH (COPY VÀO BẢNG `tab:module-results`):")
print(f" - mAP@50    : **{val_metrics.box.map50:.4f}**")
print(f" - mAP@50:95 : **{val_metrics.box.map:.4f}**")

print("\n🚀 BƯỚC 3: VẼ BIỂU ĐỒ & XUẤT ẢNH TRỰC QUAN HÓA")

# 1. Vẽ Training Curves
if 'df_logs' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(df_logs['epoch'], df_logs['train/box_loss'], label='Train Box Loss', color='#e74c3c', lw=2)
    ax1.plot(df_logs['epoch'], df_logs['val/box_loss'], label='Val Box Loss', color='#c0392b', ls='--')
    ax1.set_title('Quá trình tối ưu Loss tọa độ (Box Loss)'); ax1.set_xlabel('Epoch'); ax1.legend()

    ax2.plot(df_logs['epoch'], df_logs['metrics/mAP50(B)'], label='mAP@50', color='#2ecc71', lw=2)
    ax2.plot(df_logs['epoch'], df_logs['metrics/mAP50-95(B)'], label='mAP@50:95', color='#27ae60', ls='--')
    ax2.set_title('Đường cong tăng trưởng mAP'); ax2.set_xlabel('Epoch'); ax2.legend()

    plt.tight_layout()
    plt.savefig("/kaggle/working/report_training_curves.png", dpi=300)
    plt.close()
    print("✅ Đã vẽ biểu đồ Loss/mAP: /kaggle/working/report_training_curves.png")

# 2. Vẽ Per-class AP50 Bar Chart
cls_list = [names[i] for i in range(len(names))]
y_pos = np.arange(len(cls_list))
plt.figure(figsize=(10, 5))
bars = plt.barh(y_pos, maps50, color='#3498db', height=0.55)
plt.yticks(y_pos, cls_list); plt.xlabel('mAP@50'); plt.title('Độ chính xác mAP@50 phân theo vùng kiến thức')
plt.xlim(0, 1.08)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.01, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.savefig("/kaggle/working/report_per_class_map50.png", dpi=300)
plt.close()
print("✅ Đã vẽ biểu đồ mAP từng class: /kaggle/working/report_per_class_map50.png")

# 3. Kéo Confusion Matrix ra mặt tiền
cm_src = RUN_DIR / "confusion_matrix_normalized.png"
if cm_src.exists():
    shutil.copy2(cm_src, "/kaggle/working/report_confusion_matrix_norm.png")
    print("✅ Đã bốc Ma trận nhầm lẫn ra: /kaggle/working/report_confusion_matrix_norm.png")

print("\n🎉 HOÀN TẤT! Ở góc phải Kaggle chọn tab 'Output', ông sẽ thấy toàn bộ CSV, Weights và 3 ảnh PNG.")

🚀 BƯỚC 1: BỐC FILE LOG TỪNG EPOCH RA NGOÀI
❌ Không tìm thấy file results.csv!

🚀 BƯỚC 2: CHẠY VALIDATION TRÊN BEST.PT ĐỂ LẤY AP TỪNG CLASS
⚠️ GitHub assets check failure for https://api.github.com/repos/doclayout_yolo/assets/releases/tags/v8.1.0: 404 Not Found
⚠️ GitHub assets check failure for https://api.github.com/repos/doclayout_yolo/assets/releases/latest: 404 Not Found


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/exp1/weights/best.pt'